In [1]:
import torch
import numpy as np

In [2]:
data = torch.tensor([[[0.1, 0.3], [0.5, 0.7], [0.6, 0.8]],
                     [[0.1, 0.5], [0.2, 0.8], [0.3, 0.7]],
                     [[0.1, 0.7], [0.2, 0.8], [0.3, 0.5]],
                     [[0.2, 0.8], [0.3, 0.5], [0.1, 0.7]],
                     [[0.3, 0.7], [0.1, 0.5], [0.2, 0.8]],
                     [[0.5, 0.7], [0.2, 0.8], [0.1, 0.3]],
                     [[0.2, 0.8], [0.3, 0.7], [0.0, 0.0]],
                     [[0.2, 0.8], [0.0, 0.0], [0.0, 0.0]],
                     [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]])

print(data.shape)
print(data)

torch.Size([9, 3, 2])
tensor([[[0.1000, 0.3000],
         [0.5000, 0.7000],
         [0.6000, 0.8000]],

        [[0.1000, 0.5000],
         [0.2000, 0.8000],
         [0.3000, 0.7000]],

        [[0.1000, 0.7000],
         [0.2000, 0.8000],
         [0.3000, 0.5000]],

        [[0.2000, 0.8000],
         [0.3000, 0.5000],
         [0.1000, 0.7000]],

        [[0.3000, 0.7000],
         [0.1000, 0.5000],
         [0.2000, 0.8000]],

        [[0.5000, 0.7000],
         [0.2000, 0.8000],
         [0.1000, 0.3000]],

        [[0.2000, 0.8000],
         [0.3000, 0.7000],
         [0.0000, 0.0000]],

        [[0.2000, 0.8000],
         [0.0000, 0.0000],
         [0.0000, 0.0000]],

        [[0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.0000, 0.0000]]])


In [3]:
twos = torch.ones_like(data) * 2
threes = torch.ones_like(data) * 3

print(data.shape)
# print(data[:, :, 0:1])

data = torch.where(data == 0, 3, data)

temp_1 = torch.concat([data[:, :, 0:1], twos[:, :, 0:1]], dim=2)
temp_2 = torch.concat([data[:, :, 1:2], threes[:, :, 1:2]], dim=2)

values = None
for i in range(data.shape[1]):
    if values is None:
        values = torch.concat([temp_1[:, i:i+1, :], temp_2[:, i:i+1, :]], dim=1)
    else:
        values = torch.concat([values, temp_1[:, i:i+1, :], temp_2[:, i:i+1, :]], dim=1)

indices = values[:, :, 0].sort(stable=True)[1]
values = values[torch.arange(values.size(0)).unsqueeze(1), indices]
values = values.reshape(values.shape[0], values.shape[1] * values.shape[2])

# Indices of columns to keep
indices_to_keep = [i for i in range(values.shape[1]) if ((i + 1) % 4) - 2 != 0]
values = values[:, indices_to_keep]
values = torch.where(values == 3, 0, values)
values = torch.where(values == 2, torch.nan, values)
print(indices_to_keep)

print(values.shape)
print(values)

torch.Size([9, 3, 2])
[0, 2, 3, 4, 6, 7, 8, 10, 11]
torch.Size([9, 9])
tensor([[0.1000, 0.3000, 0.0000, 0.5000, 0.6000,    nan, 0.7000, 0.8000, 0.0000],
        [0.1000, 0.2000,    nan, 0.3000, 0.5000, 0.0000, 0.7000, 0.8000, 0.0000],
        [0.1000, 0.2000,    nan, 0.3000, 0.5000, 0.0000, 0.7000, 0.8000, 0.0000],
        [0.1000, 0.2000,    nan, 0.3000, 0.5000, 0.0000, 0.7000, 0.8000, 0.0000],
        [0.1000, 0.2000,    nan, 0.3000, 0.5000, 0.0000, 0.7000, 0.8000, 0.0000],
        [0.1000, 0.2000,    nan, 0.3000, 0.5000,    nan, 0.7000, 0.8000, 0.0000],
        [0.2000, 0.3000,    nan, 0.7000, 0.8000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.8000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])


In [14]:
import numpy as np
import open3d as o3d

def project_obb(obb_center, obb_half_extents, obb_rotation):
    # Compute the vertices of the OBB
    axes = obb_rotation.T  # Get the rotation axes
    vertices = []
    
    for sign in [-1, 1]:
        for i in range(3):
            vertex = obb_center + sign * obb_half_extents[i] * axes[:, i]
            vertices.append(vertex)
    
    return np.array(vertices)

def overlap_on_projection(proj1, proj2):
    return not (proj1[1] < proj2[0] or proj2[1] < proj1[0])

def obbs_colliding(obb1, obb2):
    # Get vertices of both OBBs
    vertices1 = project_obb(obb1['center'], obb1['half_extents'], obb1['rotation'])
    vertices2 = project_obb(obb2['center'], obb2['half_extents'], obb2['rotation'])
    
    # Get axes for projection (OBB1 axes + OBB2 axes)
    axes = np.concatenate((obb1['rotation'].T, obb2['rotation'].T), axis=1)
    
    for axis in axes.T:
        # Project vertices onto the axis
        proj1 = np.dot(vertices1, axis)
        proj2 = np.dot(vertices2, axis)
        
        # Find the min and max of the projections
        proj1_min, proj1_max = np.min(proj1), np.max(proj1)
        proj2_min, proj2_max = np.min(proj2), np.max(proj2)
        
        if not overlap_on_projection((proj1_min, proj1_max), (proj2_min, proj2_max)):
            return False  # Found a separating axis
    
    return True  # No separating axis found, boxes are colliding

# Example OBB definitions
obb1 = {
    'center': np.array([1.0, 1.0, 1.0]),
    'half_extents': np.array([0.5, 0.5, 0.5]),
    'rotation': np.eye(3)  # Identity matrix for no rotation
}

obb2 = {
    'center': np.array([0.0, 0.0, 0.0]),
    'half_extents': np.array([0.5, 0.5, 0.5]),
    'rotation': np.eye(3)  # Identity matrix for no rotation
}

# Check for collision
collision = obbs_colliding(obb1, obb2)

if collision:
    print("The OBBs are colliding.")
else:
    print("The OBBs are not colliding.")


The OBBs are colliding.


In [1]:
import numpy as np

class OBB:
    def __init__(self, center, axes, half_sizes):
        self.center = np.array(center)
        self.axes = np.array(axes)  # should be a 3x3 matrix
        self.half_sizes = np.array(half_sizes)  # length of half-sizes along each axis

def get_vertices(obb):
    """Calculate the 8 vertices of the OBB."""
    vertices = []
    for dx in [-1, 1]:
        for dy in [-1, 1]:
            for dz in [-1, 1]:
                vertex = obb.center + (dx * obb.half_sizes[0] * obb.axes[0] +
                                       dy * obb.half_sizes[1] * obb.axes[1] +
                                       dz * obb.half_sizes[2] * obb.axes[2])
                vertices.append(vertex)
    return np.array(vertices)

def overlap_on_axis(obb1, obb2, axis):
    """Check if two OBBs overlap on a given axis."""
    # Project OBB1
    proj1 = np.dot(get_vertices(obb1), axis)
    proj2 = np.dot(get_vertices(obb2), axis)
    
    min1, max1 = np.min(proj1), np.max(proj1)
    min2, max2 = np.min(proj2), np.max(proj2)
    
    return not (max1 < min2 or max2 < min1)

def check_obb_collision(obb1, obb2):
    """Check if two OBBs collide using the Separating Axis Theorem."""
    # Get the axes to check for separation
    axes = np.vstack([obb1.axes.T, obb2.axes.T])  # Combine the axes of both OBBs
    
    # Check axes from both OBBs
    for axis in axes:
        if not overlap_on_axis(obb1, obb2, axis):
            return False
    
    # Check cross products of the axes
    for i in range(3):
        for j in range(3):
            cross_axis = np.cross(obb1.axes[i], obb2.axes[j])
            if not np.all(cross_axis == 0):  # Avoid zero-length axes
                if not overlap_on_axis(obb1, obb2, cross_axis):
                    return False

    return True

# Example usage:
obb1 = OBB(center=[0, 0, 0], axes=[[1, 0, 0], [0, 1, 0], [0, 0, 1]], half_sizes=[1, 1, 1])
obb2 = OBB(center=[1, 1, 1], axes=[[0, 1, 0], [0, 0, 1], [1, 0, 0]], half_sizes=[1, 1, 1])

collision = check_obb_collision(obb1, obb2)
print("Collision Detected!" if collision else "No Collision.")

Collision Detected!


In [19]:
import itertools
from collections import defaultdict

collision_graph = defaultdict(set)

obbs = [(1, 2), (2, 3), (4, 5), (5, 6), (4, 6)]

# Detect collisions between bounding boxes
combinations = list(itertools.combinations(np.arange(7), 2))

for comb in combinations:
    collision = True if comb in obbs else False
    if collision:
        collision_graph[comb[0]].add(comb[1])
        collision_graph[comb[1]].add(comb[0])

# Step 2: Find connected components
def find_connected_components(graph):
    visited = set()
    components = []

    def dfs(node, component):
        stack = [node]
        while stack:
            current = stack.pop()
            if current not in visited:
                visited.add(current)
                component.append(current)
                stack.extend(graph[current] - visited)

    for node in graph:
        if node not in visited:
            component = []
            dfs(node, component)
            components.append(component)

    return components

# Get the groups of colliding bounding boxes
collision_groups = find_connected_components(collision_graph)

for bbox_idx in range(7):

    skip = False
    for group in collision_groups:
        if bbox_idx in group:
            skip = True

    if not skip:
        collision_groups.append([bbox_idx])

# Output the results
for i, group in enumerate(collision_groups):
    print(f"Collision Group {i + 1}: {group}")

Collision Group 1: [1, 2, 3]
Collision Group 2: [4, 6, 5]
Collision Group 3: [0]
